# 02 — 24/7 Operations & Live Dispatch (Control Tower)

The control-tower **aggregator**: reads squaring & imbalance from notebook **01** and DSR dispatch from notebook **03**, and adds per-asset dispatch, grid signals, REMIT events, and the agentic copilot. Spec: [`../specifications/02-operations-live-dispatch.md`](../specifications/02-operations-live-dispatch.md).

**Tables produced**
- `short_term_silver_grid_frequency`, `short_term_silver_market_gates`
- `short_term_gold_portfolio_balance` (balance ribbon — squaring 01 + DSR 03)
- `short_term_gold_asset_dispatch` (per-asset Mosaic AI recommended actions)
- `short_term_gold_remit_events` (Vector Search source = `raw_message`)
- `short_term_gold_copilot_recommendations` (agentic supervisor drafts)
- `short_term_gold_control_tower_summary` (header banner, P&L, alerts, handover)

Runs **after 01 and 03** in the demo job. **UC comments:** [`uc_table_comments.py`](./uc_table_comments.py).

In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC ## Preamble — read shared dimensions and upstream gold (01 + 03)

# COMMAND ----------

import os
import math
import random
import datetime as dt

from pyspark.sql import functions as F
from pyspark.sql import Row

dbutils.widgets.text("catalog", os.environ.get("DEMO_UC_CATALOG", "energy_utilities"))
dbutils.widgets.text("schema", os.environ.get("DEMO_UC_SCHEMA", "energy_trading2"))

CATALOG = dbutils.widgets.get("catalog").strip() or "energy_utilities"
SCHEMA = dbutils.widgets.get("schema").strip() or "energy_trading2"
spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")

def fq(name: str) -> str:
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"

random.seed(202)

ivals = [(r.delivery_date, r.interval_start, r.interval_index) for r in
         spark.table(fq("short_term_dim_intervals")).orderBy("interval_start").collect()]
ZONES = [r.zone_code for r in spark.table(fq("short_term_dim_zones")).select("zone_code").collect()]
DELIVERY_DATES = sorted({d for d, _, _ in ivals})
LATEST_DATE = DELIVERY_DATES[-1]
NOW_INDEX = 56
NOW_TS = dt.datetime.combine(LATEST_DATE, dt.time(0, 0)) + dt.timedelta(minutes=15 * NOW_INDEX)

def is_settled(day, idx0):
    return day < LATEST_DATE or (day == LATEST_DATE and idx0 < NOW_INDEX)

def snapshot_for(day):
    return dt.datetime.combine(day, dt.time(14, 0))

print("days:", [d.isoformat() for d in DELIVERY_DATES], "| now:", NOW_TS.isoformat())

In [ ]:
# MAGIC %md
# MAGIC ## 1. Silver — grid frequency / balancing signals + market gates

# COMMAND ----------

RESERVE_OBLIGATION = {"DE": 120.0, "NL": 60.0, "FR": 80.0, "BE": 40.0, "AT": 30.0}

freq_rows = []
for day, start, idx1 in ivals:
    idx0 = idx1 - 1
    for z in ZONES:
        dev_mhz = random.gauss(0.0, 28.0)
        # occasional larger excursions
        if random.random() < 0.04:
            dev_mhz += random.choice([-1, 1]) * random.uniform(60, 140)
        freq = 50.0 + dev_mhz / 1000.0
        afrr = round(-dev_mhz / 1000.0 * RESERVE_OBLIGATION[z] * 4.0, 2)  # proportional response
        mfrr = round(afrr * 0.5 if abs(dev_mhz) > 80 else 0.0, 2)
        if dev_mhz < -40:
            state = "CALL_UP"
        elif dev_mhz > 40:
            state = "CALL_DOWN"
        else:
            state = "NEUTRAL"
        freq_rows.append(Row(
            snapshot_ts=start,
            delivery_date=day,
            interval_start=start,
            zone_code=z,
            frequency_hz=round(freq, 4),
            frequency_deviation_mhz=round(dev_mhz, 1),
            afrr_signal_mw=afrr,
            mfrr_signal_mw=mfrr,
            reserve_obligation_mw=RESERVE_OBLIGATION[z],
            signal_state=state,
        ))
(spark.createDataFrame(freq_rows)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_silver_grid_frequency")))

# Market gates — one live snapshot per day per (market, zone).
MARKETS = ["DAY_AHEAD", "INTRADAY_AUCTION", "XBID_CONTINUOUS", "AFRR"]
gate_rows = []
for day in DELIVERY_DATES:
    snap = snapshot_for(day)
    for m in MARKETS:
        if m == "DAY_AHEAD":
            close = dt.datetime.combine(day, dt.time(12, 0))
        elif m == "INTRADAY_AUCTION":
            close = dt.datetime.combine(day, dt.time(15, 0))
        elif m == "XBID_CONTINUOUS":
            close = snap + dt.timedelta(minutes=30)  # rolling next gate
        else:  # AFRR
            close = snap + dt.timedelta(minutes=15)
        for z in ZONES:
            mins = int((close - snap).total_seconds() / 60)
            if mins < 0:
                status = "CLOSED"
            elif mins <= 15:
                status = "CLOSING"
            else:
                status = "OPEN"
            best_bid = round(60 + random.uniform(-10, 25), 2)
            gate_rows.append(Row(
                snapshot_ts=snap,
                delivery_date=day,
                market=m,
                zone_code=z,
                next_gate_close_ts=close,
                minutes_to_gate=mins,
                order_book_depth_mw=round(random.uniform(50, 800), 1),
                best_bid_eur_mwh=best_bid,
                best_ask_eur_mwh=round(best_bid + random.uniform(0.3, 2.5), 2),
                gate_status=status,
            ))
(spark.createDataFrame(gate_rows)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_silver_market_gates")))

print("grid_frequency:", spark.table(fq("short_term_silver_grid_frequency")).count())
print("market_gates:", spark.table(fq("short_term_silver_market_gates")).count())

In [ ]:
# MAGIC %md
# MAGIC ## 2. Gold — portfolio balance ribbon (aggregates 01 squaring + 03 DSR dispatch)

# COMMAND ----------

sq = (spark.table(fq("short_term_gold_squaring_actions"))
        .groupBy("delivery_date", "interval_start")
        .agg(F.round(F.sum("net_delta_mw"), 2).alias("squaring_delta_mw")))

imb = (spark.table(fq("short_term_gold_imbalance_exposure"))
        .groupBy("delivery_date", "interval_start")
        .agg(F.sum(F.coalesce(F.col("projected_cashout_eur"), F.lit(0.0))).alias("projected_cashout_eur"),
             F.max("system_balance_direction").alias("system_balance_direction")))

# DSR dispatch (03) — runs before 02, but coalesce defensively if absent.
if spark.catalog.tableExists(fq("short_term_gold_dsr_dispatch")):
    dsr = (spark.table(fq("short_term_gold_dsr_dispatch"))
            .groupBy("delivery_date", "interval_start")
            .agg(F.round(F.sum("dispatched_mw"), 2).alias("dsr_contribution_mw")))
else:
    dsr = sq.select("delivery_date", "interval_start").withColumn("dsr_contribution_mw", F.lit(0.0))

balance = (spark.table(fq("short_term_dim_intervals")).select("delivery_date", "interval_start")
    .join(sq, ["delivery_date", "interval_start"], "left")
    .join(dsr, ["delivery_date", "interval_start"], "left")
    .join(imb, ["delivery_date", "interval_start"], "left")
    .fillna(0.0, subset=["squaring_delta_mw", "dsr_contribution_mw", "projected_cashout_eur"])
    .withColumn("net_position_mw", F.round(F.col("squaring_delta_mw") + F.col("dsr_contribution_mw"), 2))
    .withColumn("system_balance_direction", F.coalesce(F.col("system_balance_direction"), F.lit("LONG")))
    .withColumn("balance_state",
        F.when(F.abs(F.col("net_position_mw")) > 60, F.lit("EXPOSED"))
         .when(F.abs(F.col("net_position_mw")) > 25, F.lit("WATCH"))
         .otherwise(F.lit("BALANCED")))
    .withColumn("snapshot_ts", F.to_timestamp(F.concat(F.col("delivery_date").cast("string"), F.lit(" 14:00:00"))))
    .select("delivery_date", "interval_start", "snapshot_ts", "net_position_mw",
            "squaring_delta_mw", "dsr_contribution_mw", "system_balance_direction",
            "projected_cashout_eur", "balance_state"))

(balance.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_gold_portfolio_balance")))

print("portfolio_balance:", spark.table(fq("short_term_gold_portfolio_balance")).count())
display(spark.table(fq("short_term_gold_portfolio_balance"))
        .filter(F.col("delivery_date") == F.lit(LATEST_DATE)).orderBy("interval_start").limit(10))

In [ ]:
# MAGIC %md
# MAGIC ## 3. Gold — per-asset dispatch (Mosaic AI recommended actions)

# COMMAND ----------

assets = spark.table(fq("short_term_dim_assets")).collect()
# Renewable live output from notebook 01's silver re-forecast.
gen_lookup = {
    (r.asset_id, r.interval_start): (r.actual_mw if r.actual_mw is not None else r.forecast_mw)
    for r in spark.table(fq("short_term_silver_generation_forecast"))
        .select("asset_id", "interval_start", "forecast_mw", "actual_mw").collect()
}

GAS_EUR_MWH = 32.0
EUA_EUR_T = 80.0

day_intervals = {}
for day, start, idx1 in ivals:
    day_intervals.setdefault(day, []).append((idx1 - 1, start))
for day in day_intervals:
    day_intervals[day].sort()

dispatch_rows = []
for day in DELIVERY_DATES:
    soc = 50.0          # battery state of charge %
    cycles_left = 2.0   # daily cycle budget
    snap = snapshot_for(day)
    for idx0, start in day_intervals[day]:
        hour = idx0 * 15 // 60
        is_peak = 8 <= hour < 20
        is_solar = 10 <= hour < 16
        is_evening = 18 <= hour < 22
        power_price = 60.0 + (40.0 if is_peak else 0.0) + (25.0 if is_evening else 0.0) + (-50.0 if is_solar else 0.0)
        for a in assets:
            soc_out = None
            cyc_out = None
            css = None
            flag = "OK"
            if a.asset_type == "BATTERY":
                if is_solar and soc < 88 and cycles_left > 0:
                    action, mw = "CHARGE", -a.nameplate_mw
                    soc = min(90.0, soc + 12.5); cycles_left -= 0.5
                elif is_evening and soc > 15 and cycles_left > 0:
                    action, mw = "DISCHARGE", a.nameplate_mw
                    soc = max(10.0, soc - 12.5); cycles_left -= 0.5
                else:
                    action, mw = "HOLD", 0.0
                if soc >= 89.5 or soc <= 10.5:
                    flag = "SOC_LIMIT"
                if cycles_left <= 0:
                    flag = "CYCLE_BUDGET"
                soc_out, cyc_out = round(soc, 1), round(cycles_left, 2)
                margin = abs(mw) * 0.25 * (power_price if mw > 0 else max(0.0, -power_price))
                out_mw = mw
            elif a.asset_type == "CCGT":
                css = power_price - (GAS_EUR_MWH / a.efficiency) - (EUA_EUR_T * a.emission_factor_tco2_mwh)
                if css > 5 and is_peak:
                    action, out_mw = "RAMP_UP", a.nameplate_mw * 0.9
                elif css < 0:
                    action, out_mw = "RAMP_DOWN", a.nameplate_mw * 0.2
                else:
                    action, out_mw = "HOLD", a.nameplate_mw * 0.5
                if action == "RAMP_DOWN":
                    flag = "MIN_RUNTIME"
                css = round(css, 2)
                margin = max(0.0, css) * out_mw * 0.25
            else:  # WIND / SOLAR — non-dispatchable
                action = "IDLE"
                out_mw = gen_lookup.get((a.asset_id, start), 0.0) or 0.0
                margin = out_mw * 0.25 * max(0.0, power_price)
            dispatch_rows.append(Row(
                delivery_date=day,
                interval_start=start,
                snapshot_ts=snap,
                asset_id=a.asset_id,
                asset_type=a.asset_type,
                current_output_mw=round(float(out_mw), 2),
                soc_pct=soc_out,
                cycle_budget_remaining=cyc_out,
                clean_spark_spread_eur=css,
                recommended_action=action,
                expected_margin_eur=round(float(margin), 2),
                policy_confidence=round(random.uniform(0.7, 0.97), 2),
                constraint_flag=flag,
            ))

(spark.createDataFrame(dispatch_rows)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_gold_asset_dispatch")))

print("asset_dispatch:", spark.table(fq("short_term_gold_asset_dispatch")).count())
display(spark.table(fq("short_term_gold_asset_dispatch"))
        .filter((F.col("delivery_date") == F.lit(LATEST_DATE)) & (F.col("asset_id") == "BATT_DE_001"))
        .orderBy("interval_start").limit(12))

In [ ]:
# MAGIC %md
# MAGIC ## 4. Gold — REMIT events (Vector Search source) + agentic copilot recommendations

# COMMAND ----------

d2, d1, d0 = DELIVERY_DATES[0], DELIVERY_DATES[1], DELIVERY_DATES[-1]

events = [
    # Hero event on the latest day — drives the control-tower demo narrative.
    Row(event_id="REMIT-0007", event_ts=dt.datetime.combine(d0, dt.time(13, 42)), delivery_date=d0,
        event_type="NUCLEAR_TRIP", affected_zone="FR", asset_or_unit="FR Nuclear Unit (illustrative)",
        capacity_mw=900.0, impact_eur_mwh=18.0, impact_horizon_hours=4.0, severity="CRITICAL",
        raw_message="URGENT MARKET MESSAGE: Unplanned outage of 900 MW nuclear generation in FR control area effective immediately, expected return in approx. 4 hours. Cross-border flows to DE/BE likely tighten; intraday prices expected to rise."),
    Row(event_id="REMIT-0006", event_ts=dt.datetime.combine(d0, dt.time(7, 15)), delivery_date=d0,
        event_type="WIND_RAMP_DOWN", affected_zone="DE", asset_or_unit="DE onshore wind fleet",
        capacity_mw=420.0, impact_eur_mwh=9.0, impact_horizon_hours=6.0, severity="WATCH",
        raw_message="Weather update: high-pressure system reduces DE onshore wind output by ~420 MW vs day-ahead schedule from 08:00; re-forecast issued."),
    Row(event_id="REMIT-0005", event_ts=dt.datetime.combine(d0, dt.time(11, 5)), delivery_date=d0,
        event_type="SOLAR_SHOCK", affected_zone="DE", asset_or_unit="DE solar PV",
        capacity_mw=300.0, impact_eur_mwh=-22.0, impact_horizon_hours=2.0, severity="WATCH",
        raw_message="Clear-sky midday solar surge across DE/NL pushing intraday prices negative for the 12:00-14:00 window; charging/absorption opportunity."),
    Row(event_id="REMIT-0004", event_ts=dt.datetime.combine(d1, dt.time(18, 30)), delivery_date=d1,
        event_type="INTERCONNECTOR_OUTAGE", affected_zone="BE", asset_or_unit="BE-FR interconnector",
        capacity_mw=1000.0, impact_eur_mwh=14.0, impact_horizon_hours=8.0, severity="CRITICAL",
        raw_message="Planned-to-forced extension: BE-FR interconnector capacity reduced by 1000 MW into the evening peak; scarcity risk in BE."),
    Row(event_id="REMIT-0003", event_ts=dt.datetime.combine(d1, dt.time(9, 20)), delivery_date=d1,
        event_type="THERMAL_TRIP", affected_zone="DE", asset_or_unit="DE hard-coal unit (illustrative)",
        capacity_mw=600.0, impact_eur_mwh=11.0, impact_horizon_hours=5.0, severity="WATCH",
        raw_message="Unplanned trip of ~600 MW thermal unit in DE; balancing demand expected to increase."),
    Row(event_id="REMIT-0002", event_ts=dt.datetime.combine(d2, dt.time(17, 10)), delivery_date=d2,
        event_type="WIND_RAMP_DOWN", affected_zone="NL", asset_or_unit="NL wind",
        capacity_mw=250.0, impact_eur_mwh=7.0, impact_horizon_hours=4.0, severity="INFO",
        raw_message="Minor NL wind ramp-down ~250 MW into the evening; limited price impact expected."),
]
(spark.createDataFrame(events)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_gold_remit_events")))

# Copilot recommendations — agentic supervisor drafts, grounded in events + live tables.
recs = [
    Row(recommendation_id="COPILOT-0007", generated_ts=dt.datetime.combine(d0, dt.time(13, 43)), delivery_date=d0,
        linked_event_id="REMIT-0007",
        recommendation="Discharge BATT_DE_001 80 MW into 14:00-18:00 peak and bid 60 MW DSR (MFRR) into balancing.",
        rationale="900 MW FR nuclear trip (REMIT-0007) tightens FR->DE flows; DE intraday modeled +€18/MWh for ~4h. Battery SoC 75% and 1.5 cycles remain; ~150 MW DSR flex-up is prequalified in DE. Capturing the uplift across 4h is worth an estimated €31k.",
        expected_pnl_impact_eur=31000.0, confidence=0.86,
        agent_trace="classify(REMIT-0007)->retrieve(3 precedents via Vector Search)->query(balance,asset_dispatch,dsr_bid_stack via Genie)", status="PROPOSED"),
    Row(recommendation_id="COPILOT-0005", generated_ts=dt.datetime.combine(d0, dt.time(11, 6)), delivery_date=d0,
        linked_event_id="REMIT-0005",
        recommendation="Charge BATT_DE_001 and signal EV/BTM DSR to absorb during the 12:00-14:00 negative-price window.",
        rationale="Midday solar surge (REMIT-0005) drives DE intraday to ~-€22/MWh; absorbing 100 MW battery + ~40 MW DSR avoids paying to stay long and refills SoC for the evening peak.",
        expected_pnl_impact_eur=8500.0, confidence=0.79,
        agent_trace="classify(REMIT-0005)->retrieve(2 precedents)->query(imbalance_exposure,dsr_availability)", status="PROPOSED"),
    Row(recommendation_id="COPILOT-0004", generated_ts=dt.datetime.combine(d1, dt.time(18, 31)), delivery_date=d1,
        linked_event_id="REMIT-0004",
        recommendation="Hold CCGT_DE_001 at 90% and pre-position 50 MW MFRR bid for the BE-driven evening scarcity.",
        rationale="BE-FR interconnector loss (REMIT-0004) raises evening scarcity risk; clean spark spread is positive at peak, so keeping the CCGT high and offering reserve captures balancing upside.",
        expected_pnl_impact_eur=12400.0, confidence=0.74,
        agent_trace="classify(REMIT-0004)->retrieve(1 precedent)->query(asset_dispatch,market_gates)", status="PROPOSED"),
]
(spark.createDataFrame(recs)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_gold_copilot_recommendations")))

print("remit_events:", spark.table(fq("short_term_gold_remit_events")).count())
print("copilot_recommendations:", spark.table(fq("short_term_gold_copilot_recommendations")).count())

In [ ]:
# MAGIC %md
# MAGIC ## 5. Gold — control-tower summary (header, P&L, alerts, handover)

# COMMAND ----------

from pyspark.sql.window import Window

balance = spark.table(fq("short_term_gold_portfolio_balance"))
disp = spark.table(fq("short_term_gold_asset_dispatch"))
remit = spark.table(fq("short_term_gold_remit_events"))
recs = spark.table(fq("short_term_gold_copilot_recommendations"))

# Net position / balance state at each day's 14:00 snapshot interval.
balance_at_snap = (balance.filter(F.col("interval_start") == F.col("snapshot_ts"))
    .select("delivery_date", "snapshot_ts", "net_position_mw", "balance_state"))

pnl = disp.groupBy("delivery_date").agg(F.round(F.sum("expected_margin_eur"), 2).alias("pnl_since_shift_eur"))
alerts = (balance.filter(F.col("balance_state") == "EXPOSED")
    .groupBy("delivery_date").agg(F.count("*").alias("n_open_alerts")))
crit = (remit.filter(F.col("severity") == "CRITICAL")
    .groupBy("delivery_date").agg(F.count("*").alias("n_critical_events")))
actionable = (disp.filter((F.col("interval_start") == F.col("snapshot_ts")) &
                          (~F.col("recommended_action").isin("HOLD", "IDLE")))
    .groupBy("delivery_date").agg(F.count("*").alias("n_assets_actionable")))

w = Window.partitionBy("delivery_date").orderBy(F.col("confidence").desc())
top_rec = (recs.filter(F.col("status") == "PROPOSED")
    .withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1)
    .select("delivery_date", F.col("recommendation_id").alias("top_copilot_recommendation_id")))

summary = (balance_at_snap
    .join(pnl, "delivery_date", "left")
    .join(alerts, "delivery_date", "left")
    .join(crit, "delivery_date", "left")
    .join(actionable, "delivery_date", "left")
    .join(top_rec, "delivery_date", "left")
    .fillna(0, subset=["pnl_since_shift_eur", "n_open_alerts", "n_critical_events", "n_assets_actionable"])
    .withColumn("shift_start_ts", F.to_timestamp(F.concat(F.col("delivery_date").cast("string"), F.lit(" 06:00:00"))))
    .withColumn("headline",
        F.when(F.col("n_critical_events") > 0, F.lit("CRITICAL event live — review copilot recommendation and act"))
         .when(F.col("balance_state") == "EXPOSED", F.lit("Portfolio exposed — rebalance before delivery"))
         .otherwise(F.lit("Book balanced — monitoring grid and gates")))
    .withColumn("handover_notes",
        F.when(F.col("n_critical_events") > 0, F.lit("Carry open copilot action into next shift; confirm DSR + battery dispatch settled."))
         .otherwise(F.lit("No open critical items. Standard monitoring.")))
    .select("delivery_date", "snapshot_ts", "headline", "pnl_since_shift_eur", "net_position_mw",
            "balance_state", "n_open_alerts", "n_critical_events", "n_assets_actionable",
            "top_copilot_recommendation_id", "shift_start_ts", "handover_notes"))

(summary.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_gold_control_tower_summary")))

display(spark.table(fq("short_term_gold_control_tower_summary")).orderBy(F.col("delivery_date").desc()))

In [ ]:
# MAGIC %md
# MAGIC ## 6. Row counts + Unity Catalog comments

# COMMAND ----------

for t in [
    "short_term_silver_grid_frequency",
    "short_term_silver_market_gates",
    "short_term_gold_portfolio_balance",
    "short_term_gold_asset_dispatch",
    "short_term_gold_remit_events",
    "short_term_gold_copilot_recommendations",
    "short_term_gold_control_tower_summary",
]:
    print(f"  {t:42s}  {spark.table(fq(t)).count():>10,} rows")

# COMMAND ----------

from pathlib import Path

_uc_paths = []
try:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _uc_paths.append(Path(_nb).parent / "uc_table_comments.py")
except Exception:
    pass
_uc_paths.append(Path.cwd() / "uc_table_comments.py")

_uc_py = next((p for p in _uc_paths if p.is_file()), None)
if _uc_py is None:
    raise FileNotFoundError("uc_table_comments.py not found next to this notebook.")

exec(_uc_py.read_text(), globals())
apply_short_term_notebook_02_comments(spark, CATALOG, SCHEMA)
print("UC comments applied for notebook 02.")